# 🔍 Trend Finder — Gemini LLM

This notebook identifies the **top 3 trending topics per niche** by:
1. Loading the clustered Reddit data
2. Scoring each cluster by engagement (score, comments, upvote ratio)
3. Sending the most important clusters to **Gemini** as context
4. Asking Gemini to generate a **trend title**, **description**, and **references** for each trend

**Niches covered:** `technology` · `worldnews` · `science` · `gaming` · `movies`

## 1. Install Dependencies

In [1]:
!pip install google-generativeai python-dotenv pandas numpy scikit-learn tqdm

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
  Using cached googleapis_common_protos-1.73.0-py3-none-any.whl.metadata (9.4 kB)
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
  Using cached google_api_core-2.30.0-py3-none-any.whl.metadata (3.1 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   -----------

In [16]:
!pip install spacy
!python -m spacy download en_core_web_sm

   ---------------------------------------- 0.0/14.4 MB ? eta -:--:--
   -- ------------------------------------- 0.8/14.4 MB 8.4 MB/s eta 0:00:02
   ---------- ----------------------------- 3.7/14.4 MB 13.1 MB/s eta 0:00:01
   ---------- ----------------------------- 3.7/14.4 MB 13.1 MB/s eta 0:00:01
   ------------------- -------------------- 7.1/14.4 MB 9.7 MB/s eta 0:00:01
   ----------------------- ---------------- 8.4/14.4 MB 9.2 MB/s eta 0:00:01
   -------------------------- ------------- 9.7/14.4 MB 8.8 MB/s eta 0:00:01
   -------------------------------- ------- 11.8/14.4 MB 8.8 MB/s eta 0:00:01
   ------------------------------------- -- 13.4/14.4 MB 8.6 MB/s eta 0:00:01
   ---------------------------------------- 14.4/14.4 MB 8.3 MB/s  0:00:01
   ---------------------------------------- 0.0/656.5 kB ? eta -:--:--
   ---------------------------------------- 656.5/656.5 kB 5.9 MB/s  0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------

## 2. Import Libraries

In [1]:
import os
import json
import time
import textwrap
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import google.generativeai as genai
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries imported!')

C:\Users\tfahi\AppData\Local\Temp\ipykernel_51764\879979439.py:8: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


✅ All libraries imported!


## 3. Load API Key from .env

Create a `.env` file in the same folder as this notebook with:
```
GEMINI_API_KEY=your_api_key_here
```

In [12]:
# Load .env
load_dotenv()
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

if not GEMINI_API_KEY:
    raise ValueError('❌ GEMINI_API_KEY not found. Make sure your .env file exists and has the key.')

# Configure Gemini
genai.configure(api_key=GEMINI_API_KEY)

# ✏️ Choose your Gemini model
# Options: 'gemini-2.5-flash' (faster, cheaper) | 'gemini-2.5-pro' (more powerful)
GEMINI_MODEL = 'gemini-2.5-flash-lite'

model = genai.GenerativeModel(GEMINI_MODEL)

print(f'✅ Gemini configured!')
print(f'   Model : {GEMINI_MODEL}')
print(f'   Key   : {GEMINI_API_KEY[:8]}...{GEMINI_API_KEY[-4:]}')

✅ Gemini configured!
   Model : gemini-2.5-flash-lite
   Key   : AIzaSyDN...Pn0I


## 4. Load Clustered Reddit Data

In [3]:
# Load the full clustered dataset produced by kmeans_reddit.ipynb
df = pd.read_csv('technology_with_clusters.csv')

# Basic cleanup
df['score']        = pd.to_numeric(df['score'], errors='coerce').fillna(0)
df['num_comments'] = pd.to_numeric(df['num_comments'], errors='coerce').fillna(0)
df['upvote_ratio'] = pd.to_numeric(df['upvote_ratio'], errors='coerce').fillna(0)
df['title']        = df['title'].fillna('').astype(str)
df['permalink']    = df['permalink'].fillna('').astype(str)

NICHES = sorted(df['niche'].unique().tolist())

print(f'✅ Dataset loaded: {len(df):,} posts')
print(f'   Available niches  : {NICHES}')
print(f'   Clusters present  : {sorted(df["cluster"].unique().tolist())}')
print(f'   Date range        : {df["timestamp_utc"].min()} → {df["timestamp_utc"].max()}')
df.head(3)

✅ Dataset loaded: 1,417 posts
   Available niches  : ['technology']
   Clusters present  : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
   Date range        : 2026-03-25 08:56:14 → 2026-04-24 08:19:06


,post_id,niche,title,author,score,upvote_ratio,num_comments,timestamp_utc,permalink,url,selftext,full_text,text_translated,cluster
0,1skp97a,technology,Man Who Threw Molotov Cocktail At Sam Altman’s...,Amentet,68149,0.91,757,2026-04-13 21:50:09,https://reddit.com/r/technology/comments/1skp9...,https://theonion.com/man-who-threw-molotov-coc...,NaN,Man Who Threw Molotov Cocktail At Sam Altman’s...,Man Who Threw Molotov Cocktail At Sam Altman’s...,3
1,1smfq4p,technology,"Ticketmaster is an illegal monopoly, jury rule...",MarvelsGrantMan136,59637,0.98,1091,2026-04-15 19:10:11,https://reddit.com/r/technology/comments/1smfq...,https://www.theverge.com/policy/912689/live-na...,NaN,"Ticketmaster is an illegal monopoly, jury rule...","Ticketmaster is an illegal monopoly, jury rule...",3
2,1s6vuih,technology,Epic Games Layoffs Included Terminally Ill Fat...,Turbostrider27,36676,0.93,2351,2026-03-29 14:00:10,https://reddit.com/r/technology/comments/1s6vu...,https://www.thegamer.com/epic-games-layoff-ter...,NaN,Epic Games Layoffs Included Terminally Ill Fat...,Epic Games Layoffs Included Terminally Ill Fat...,3


## 4b. Parse Weeks from Timestamp

In [4]:
# Parse timestamp and assign week numbers
df['timestamp_utc'] = pd.to_datetime(df['timestamp_utc'])

# Assign ISO week number and year
df['week_number'] = df['timestamp_utc'].dt.isocalendar().week.astype(int)
df['year']        = df['timestamp_utc'].dt.isocalendar().year.astype(int)
df['week_label']  = df.apply(
    lambda r: f"Week {r['week_number']} ({r['timestamp_utc'].strftime('%b %d')} – "
              f"{(r['timestamp_utc'] + pd.Timedelta(days=6 - r['timestamp_utc'].weekday())).strftime('%b %d, %Y')})",
    axis=1
)

# Build a clean week summary
week_summary = (
    df.groupby(['year', 'week_number'])
    .agg(
        post_count    = ('post_id', 'count'),
        date_start    = ('timestamp_utc', 'min'),
        date_end      = ('timestamp_utc', 'max'),
    )
    .reset_index()
    .sort_values(['year', 'week_number'])
)

week_summary['label'] = week_summary.apply(
    lambda r: f"Week {r['week_number']}  |  "
              f"{r['date_start'].strftime('%b %d')} – {r['date_end'].strftime('%b %d, %Y')}  |  "
              f"{r['post_count']:,} posts",
    axis=1
)

print(f'📅 Available weeks in dataset:\n')
for i, row in week_summary.iterrows():
    print(f"  [{row['week_number']}]  {row['label']}")

AVAILABLE_WEEKS = week_summary['week_number'].tolist()
print(f'\n✅ Total weeks available: {len(AVAILABLE_WEEKS)}')

📅 Available weeks in dataset:

  [13]  Week 13  |  Mar 25 – Mar 29, 2026  |  148 posts
  [14]  Week 14  |  Mar 30 – Apr 05, 2026  |  223 posts
  [15]  Week 15  |  Apr 06 – Apr 12, 2026  |  394 posts
  [16]  Week 16  |  Apr 13 – Apr 19, 2026  |  357 posts
  [17]  Week 17  |  Apr 20 – Apr 24, 2026  |  295 posts

✅ Total weeks available: 5


## 5. Helper Functions

These functions handle:
- **Scoring clusters** by engagement to find the most important ones
- **Building the prompt** that gets sent to Gemini
- **Parsing Gemini's response** into structured trend objects

In [5]:
def score_clusters(df_niche, top_n_clusters=3):
    """
    Score each cluster by a weighted engagement metric and return
    the top N most important cluster IDs.

    Importance = (avg_score * 0.5) + (avg_comments * 0.3) + (avg_upvote_ratio * 100 * 0.2)
    Weighted so that highly upvoted AND discussed clusters rank highest.
    """
    cluster_stats = []

    for cluster_id, group in df_niche.groupby('cluster'):
        importance = (
            group['score'].mean()        * 0.5 +
            group['num_comments'].mean() * 0.3 +
            group['upvote_ratio'].mean() * 100 * 0.2
        )
        cluster_stats.append({
            'cluster_id'      : cluster_id,
            'importance_score': round(importance, 2),
            'post_count'      : len(group),
            'avg_score'       : round(group['score'].mean(), 1),
            'avg_comments'    : round(group['num_comments'].mean(), 1),
            'avg_upvote_ratio': round(group['upvote_ratio'].mean(), 3),
        })

    stats_df = pd.DataFrame(cluster_stats).sort_values('importance_score', ascending=False)
    top_ids  = stats_df.head(top_n_clusters)['cluster_id'].tolist()
    return stats_df, top_ids


def extract_keywords(texts, top_n=8):
    """Extract top keywords from a list of texts."""
    all_words = []
    for text in texts:
        words = str(text).lower().split()
        words = [
            w for w in words
            if w not in ENGLISH_STOP_WORDS
            and len(w) > 3
            and not w.startswith(('http', '@', '#', 'www'))
            and w.isalpha()
        ]
        all_words.extend(words)
    return [word for word, _ in Counter(all_words).most_common(top_n)]


import spacy

# Load once globally — not inside the function
nlp = spacy.load('en_core_web_sm')


def extract_entities(texts, top_n=10):
    """
    Extract top named entities from a list of texts.
    Focuses on ORG (companies), PRODUCT, PERSON, GPE (countries/cities).
    Returns a dict grouped by entity type.
    """
    entity_counts = {
        'ORG'    : Counter(),   # companies, organizations
        'PRODUCT': Counter(),   # products, services
        'PERSON' : Counter(),   # people
        'GPE'    : Counter(),   # countries, cities
    }

    for text in texts:
        doc = nlp(str(text)[:512])  # cap at 512 chars per text for speed
        for ent in doc.ents:
            if ent.label_ in entity_counts:
                entity_counts[ent.label_][ent.text.strip()] += 1

    # Return top N per category, filtering out single-character noise
    result = {}
    for label, counter in entity_counts.items():
        top = [
            (name, count) for name, count in counter.most_common(top_n)
            if len(name) > 2
        ]
        if top:
            result[label] = top

    return result


def build_cluster_context(df_niche, cluster_id, top_posts_n=8):
    from sklearn.feature_extraction.text import CountVectorizer

    group     = df_niche[df_niche['cluster'] == cluster_id]
    top_posts = group.sort_values('score', ascending=False).head(top_posts_n)

    # ── Keywords ──────────────────────────────────────────────
    keywords = extract_keywords(group['text_translated'].tolist(), top_n=15)

    # ── Bigrams ───────────────────────────────────────────────
    try:
        vec = CountVectorizer(
            ngram_range=(2, 3),
            stop_words='english',
            max_features=10,
            min_df=2
        )
        vec.fit(group['title'].tolist())
        bigrams = list(vec.vocabulary_.keys())
    except Exception:
        bigrams = []

    # ── Sentiment ─────────────────────────────────────────────
    negative_words = {'ban', 'fail', 'broken', 'worst', 'hate', 'scam',
                      'fraud', 'wrong', 'bad', 'terrible', 'dangerous',
                      'illegal', 'blocked', 'cancelled', 'delayed', 'fined'}
    titles_lower = group['title'].str.lower()
    negative_pct = titles_lower.apply(
        lambda t: any(w in t for w in negative_words)
    ).mean()

    sentiment = (
        'mostly negative/critical' if negative_pct > 0.5
        else 'mostly positive/excited' if negative_pct < 0.2
        else 'mixed sentiment'
    )

    # ── NER ───────────────────────────────────────────────────
    # Run NER on titles (shorter, cleaner than full text)
    entities = extract_entities(group['title'].tolist(), top_n=8)

    # Format entities for the prompt
    entity_lines = []
    label_map = {
        'ORG'    : '🏢 Companies/Orgs',
        'PRODUCT': '📦 Products/Services',
        'PERSON' : '👤 People',
        'GPE'    : '🌍 Places',
    }
    for label, items in entities.items():
        names = ', '.join([f'{name} ({count}x)' for name, count in items])
        entity_lines.append(f'  {label_map[label]}: {names}')

    entity_block = '\n'.join(entity_lines) if entity_lines else '  None detected'

    # ── Build context block ───────────────────────────────────
    lines = [
        f'--- CLUSTER {cluster_id} ---',
        f'Post count         : {len(group)}',
        f'Avg score          : {group["score"].mean():.0f}',
        f'Avg comments       : {group["num_comments"].mean():.0f}',
        f'Avg upvote ratio   : {group["upvote_ratio"].mean():.2f}',
        f'Community sentiment: {sentiment} ({negative_pct:.0%} posts are critical)',
        f'',
        f'Recurring keywords : {", ".join(keywords)}',
        f'Recurring phrases  : {", ".join(bigrams) if bigrams else "n/a"}',
        f'',
        f'Named entities mentioned across posts (frequency in brackets):',
        entity_block,
        f'',
        f'Top posts by engagement (use these URLs as references only):',
    ]

    for i, (_, row) in enumerate(top_posts.iterrows(), 1):
        lines.append(
            f'  {i}. [Score: {int(row["score"]):,} | '
            f'Comments: {int(row["num_comments"]):,}] '
            f'{row["title"][:100]}'
        )
        lines.append(f'     URL: {row["permalink"]}')

    return '\n'.join(lines)


def build_prompt(niche, cluster_contexts):
    context_block = '\n\n'.join(cluster_contexts)

    prompt = f"""You are a social media trend analyst specializing in identifying emerging patterns on Reddit.

Below is cluster data from r/{niche}. Each cluster is a group of posts that Reddit users have been repeatedly discussing over the past 7 days.

IMPORTANT DISTINCTION:
- A NEWS EVENT is a single specific thing that happened (e.g. "Company X fined $1M")
- A TREND is a PATTERN — a topic, sentiment, or concern that appears repeatedly across many posts and is gaining momentum (e.g. "Growing frustration with corporate monopolies affecting everyday consumers")

Your job is to identify TRENDS, not summarize news events.

REDDIT CLUSTER DATA (last 7 days):
{context_block}

Based on the patterns you see ACROSS MULTIPLE POSTS in the clusters above, identify the TOP 3 TRENDS in r/{niche}.

A good trend answer:
✅ Describes a pattern seen across many posts, not a single event
✅ Explains WHY this topic keeps coming up and what it signals about community sentiment
✅ Is forward-looking — what does this pattern suggest is building or shifting?
✅ Mentions specific companies, products, or people from the Named Entities if they are central to the trend
✅ Uses specific posts as evidence/references, not as the trend itself

For each trend provide:
1. TREND TITLE — short, pattern-based, max 10 words
2. DESCRIPTION — 3 sentences: (a) what pattern is emerging, (b) why the community keeps discussing it, (c) what it signals
3. KEY ENTITIES — list any companies, products, or people that are central to this trend (leave empty if none)
4. REFERENCES — 2-3 URLs from the top posts listed above as evidence

Respond ONLY in this JSON format, no markdown, no code fences:
{{
  "niche": "{niche}",
  "trends": [
    {{
      "rank": 1,
      "title": "Trend title here",
      "description": "Description here.",
      "key_entities": {{
        "companies" : ["e.g. Google", "Microsoft"],
        "products"  : ["e.g. ChatGPT"],
        "people"    : ["e.g. Elon Musk"]
      }},
      "references": [
        "https://reddit.com/...",
        "https://reddit.com/..."
      ]
    }},
    {{
      "rank": 2,
      "title": "...",
      "description": "...",
      "key_entities": {{"companies": [], "products": [], "people": []}},
      "references": ["..."]
    }},
    {{
      "rank": 3,
      "title": "...",
      "description": "...",
      "key_entities": {{"companies": [], "products": [], "people": []}},
      "references": ["..."]
    }}
  ]
}}"""
    return prompt


def call_gemini(prompt, retries=3, delay=5):
    """
    Call Gemini API with retry logic.
    Returns the raw text response.
    """
    for attempt in range(1, retries + 1):
        try:
            response = model.generate_content(prompt)
            return response.text.strip()
        except Exception as e:
            print(f'  ⚠️  Attempt {attempt}/{retries} failed: {e}')
            if attempt < retries:
                time.sleep(delay)
    return None


def parse_gemini_response(raw_response):
    """
    Parse Gemini's JSON response into a Python dict.
    Strips markdown fences if Gemini adds them despite instructions.
    """
    if not raw_response:
        return None
    try:
        # Strip markdown fences just in case
        clean = raw_response.replace('```json', '').replace('```', '').strip()
        return json.loads(clean)
    except json.JSONDecodeError as e:
        print(f'  ❌ JSON parse error: {e}')
        print(f'  Raw response snippet: {raw_response[:300]}')
        return None


print('✅ Helper functions defined!')

✅ Helper functions defined!


## 6. Configuration

In [6]:
# ============================================================
# ✏️  CONFIGURATION — adjust as needed
# ============================================================

# How many top clusters to send to Gemini per niche
# (these are the highest engagement clusters)
TOP_N_CLUSTERS = 3

# How many top posts to include per cluster in the prompt
TOP_POSTS_PER_CLUSTER = 8

# Output file
OUTPUT_FILE = 'reddit_trends_llm.json'

# All 5 niches to process in the full pipeline
ALL_NICHES = ['technology', 'worldnews', 'science', 'gaming', 'movies']

# ============================================================
print('✅ Configuration set!')
print(f'   Top clusters per niche : {TOP_N_CLUSTERS}')
print(f'   Top posts per cluster  : {TOP_POSTS_PER_CLUSTER}')
print(f'   Niches to process      : {ALL_NICHES}')

✅ Configuration set!
   Top clusters per niche : 3
   Top posts per cluster  : 8
   Niches to process      : ['technology', 'worldnews', 'science', 'gaming', 'movies']


---
# 🗓️ PART A — Single Niche, Single Week Experimentation

**Cell A1 — Week + Niche Selector**

In [7]:
# ============================================================
# ✏️  SELECT YOUR NICHE AND WEEK HERE
# ============================================================
NICHE        = 'technology'   # options: 'technology', 'worldnews', 'science', 'gaming', 'movies'
SELECTED_WEEK = 16            # ✏️ pick a week number from the list printed above
# ============================================================

df_filtered = df[
    (df['niche'] == NICHE) &
    (df['week_number'] == SELECTED_WEEK)
].reset_index(drop=True)

# Week label for display
week_row   = week_summary[week_summary['week_number'] == SELECTED_WEEK].iloc[0]
week_label = f"Week {SELECTED_WEEK} ({week_row['date_start'].strftime('%b %d')} – {week_row['date_end'].strftime('%b %d, %Y')})"

print(f'🎯 Niche    : r/{NICHE}')
print(f'📅 Week     : {week_label}')
print(f'📊 Posts    : {len(df_filtered):,}')
print(f'🔢 Clusters : {sorted(df_filtered["cluster"].unique().tolist())}')

if len(df_filtered) == 0:
    print(f'\n⚠️  No posts found for r/{NICHE} in {week_label}.')
    print('    Try a different week or niche.')

🎯 Niche    : r/technology
📅 Week     : Week 16 (Apr 13 – Apr 19, 2026)
📊 Posts    : 357
🔢 Clusters : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


**Cell A2 — Cluster Importance for Selected Week**

In [8]:
stats_df, top_cluster_ids = score_clusters(df_filtered, top_n_clusters=TOP_N_CLUSTERS)

print(f'📊 Cluster Importance — r/{NICHE} | {week_label}\n')
print(stats_df.to_string(index=False))
print(f'\n✅ Top {TOP_N_CLUSTERS} clusters selected for Gemini: {top_cluster_ids}')

📊 Cluster Importance — r/technology | Week 16 (Apr 13 – Apr 19, 2026)

 cluster_id  importance_score  post_count  avg_score  avg_comments  avg_upvote_ratio
          3           2252.65          38     4403.6         109.3             0.903
          5           1924.90          36     3662.8         253.2             0.875
          2           1511.24          19     2914.0         122.2             0.879
          6           1452.00          28     2780.4         151.7             0.815
         10           1375.13          25     2523.4         322.1             0.840
          8           1291.13          47     2435.7         183.1             0.916
          4           1168.45          67     2184.2         204.6             0.749
          0            639.85          26     1187.7          94.5             0.883
          9            548.07          12     1015.7          81.0             0.797
          1            220.29          31      382.0          49.6             

**Cell A3 — Send to Gemini and Display**

In [9]:
# Build prompt
cluster_contexts = [
    build_cluster_context(df_filtered, cid, top_posts_n=TOP_POSTS_PER_CLUSTER)
    for cid in top_cluster_ids
]
prompt = build_prompt(NICHE, cluster_contexts)

print(f'🚀 Sending r/{NICHE} | {week_label} to Gemini ({GEMINI_MODEL})...\n')
raw_response = call_gemini(prompt)

# Parse and display
result = parse_gemini_response(raw_response)

if result:
    print(f'\n🔥 TOP 3 TRENDS — r/{NICHE.upper()} | {week_label}\n')
    print('=' * 70)

    for trend in result.get('trends', []):
        print(f"\n#{trend['rank']} — {trend['title']}")
        print(f"\n  {trend['description']}")

        entities  = trend.get('key_entities', {})
        companies = entities.get('companies', [])
        products  = entities.get('products', [])
        people    = entities.get('people', [])

        if any([companies, products, people]):
            print(f"\n  🏷️  Key Entities:")
            if companies : print(f"     🏢 Companies : {', '.join(companies)}")
            if products  : print(f"     📦 Products  : {', '.join(products)}")
            if people    : print(f"     👤 People    : {', '.join(people)}")

        print(f"\n  📎 References:")
        for ref in trend.get('references', []):
            print(f"     • {ref}")
        print()

    print('=' * 70)
else:
    print('❌ Could not parse response. Check raw output:')
    print(raw_response)

🚀 Sending r/technology | Week 16 (Apr 13 – Apr 19, 2026) to Gemini (gemini-2.5-flash)...


🔥 TOP 3 TRENDS — r/TECHNOLOGY | Week 16 (Apr 13 – Apr 19, 2026)


#1 — Corporate Accountability and Ethical Tech Practices Under Fire

  A pattern of intense public scrutiny and growing frustration with the power and ethical conduct of large corporations, including tech giants, is emerging. The community is increasingly vocal about issues ranging from monopolies and consumer rights to the broader societal impact and surveillance capabilities of these entities. This signals a building demand for greater regulation, transparency, and corporate responsibility in the tech sector and beyond.

  🏷️  Key Entities:
     🏢 Companies : Live Nation-Ticketmaster, Meta, Missouri Town Council, Paramount-Warner, Toshiba, Madison Square Garden

  📎 References:
     • https://reddit.com/r/technology/comments/1smfq4p/ticketmaster_is_an_illegal_monopoly_jury_rules/
     • https://reddit.com/r/technology/comments/1s

---
 
# 📆 PART B — Week-by-Week Comparison (Single Niche, All Weeks)

**Cell B1 — Run All Weeks for One Niche**

In [13]:
# ✏️ Set niche for week-by-week comparison
NICHE_WEEKLY = 'technology'

weekly_results = []

print(f'🔍 Running week-by-week trend analysis for r/{NICHE_WEEKLY}...\n')
print('=' * 70)

for _, week_row in week_summary.iterrows():
    wnum  = int(week_row['week_number'])
    wlabel = f"Week {wnum} ({week_row['date_start'].strftime('%b %d')} – {week_row['date_end'].strftime('%b %d, %Y')})"

    df_w = df[
        (df['niche'] == NICHE_WEEKLY) &
        (df['week_number'] == wnum)
    ].reset_index(drop=True)

    if len(df_w) < 10:
        print(f'  ⚠️  {wlabel} — only {len(df_w)} posts, skipping.')
        continue

    print(f'\n📅 {wlabel} ({len(df_w):,} posts)')

    stats, top_ids = score_clusters(df_w, top_n_clusters=TOP_N_CLUSTERS)
    contexts       = [
        build_cluster_context(df_w, cid, top_posts_n=TOP_POSTS_PER_CLUSTER)
        for cid in top_ids
    ]
    prompt = build_prompt(NICHE_WEEKLY, contexts)

    print(f'  Calling Gemini...')
    raw    = call_gemini(prompt)
    result = parse_gemini_response(raw)

    if result:
        result['week_number'] = wnum
        result['week_label']  = wlabel
        result['post_count']  = len(df_w)
        weekly_results.append(result)
        titles = [f"#{t['rank']} {t['title']}" for t in result.get('trends', [])]
        print(f'  ✅ Trends: {" | ".join(titles)}')
    else:
        print(f'  ❌ Failed to parse response for {wlabel}')

    time.sleep(4)  # Gemini rate limit

print(f'\n{"="*70}')
print(f'✅ Done! {len(weekly_results)} weeks processed for r/{NICHE_WEEKLY}')

🔍 Running week-by-week trend analysis for r/technology...


📅 Week 13 (Mar 25 – Mar 29, 2026) (148 posts)
  Calling Gemini...
  ✅ Trends: #1 AI's Growing Impact on Jobs and Content Creation | #2 Concerns Over Data Privacy and Security Breaches | #3 Platform Accountability and Regulatory Scrutiny

📅 Week 14 (Mar 30 – Apr 05, 2026) (223 posts)
  Calling Gemini...
  ✅ Trends: #1 Geopolitical Tensions Impacting Global Tech Infrastructure | #2 Layoffs, H-1B Visas, and Corporate Restructuring Concerns | #3 Challenges and Delays in Data Center Expansion

📅 Week 15 (Apr 06 – Apr 12, 2026) (394 posts)
  Calling Gemini...
  ✅ Trends: #1 OpenAI's Regulatory Scrutiny and Internal Conflicts | #2 Geopolitical Tensions Impacting Digital Infrastructure | #3 Microsoft's Evolving Windows Ecosystem and AI Integration

📅 Week 16 (Apr 13 – Apr 19, 2026) (357 posts)
  Calling Gemini...
  ✅ Trends: #1 AI Safety Concerns and Unpredictable AI Behavior | #2 Monopoly Power and Corporate Overreach in Tech | #3 Te

**Cell B2 — Display Week-by-Week Comparison**

In [14]:
print(f'\n📆 WEEK-BY-WEEK TRENDS — r/{NICHE_WEEKLY.upper()}\n')
print('=' * 70)

for week_result in weekly_results:
    print(f'\n{"─"*70}')
    print(f'📅  {week_result["week_label"]}  |  {week_result["post_count"]:,} posts analysed')
    print(f'{"─"*70}')

    for trend in week_result.get('trends', []):
        print(f"\n  #{trend['rank']} — {trend['title']}")
        print(f"  {trend['description']}")

        entities  = trend.get('key_entities', {})
        companies = entities.get('companies', [])
        products  = entities.get('products', [])
        people    = entities.get('people', [])

        if any([companies, products, people]):
            print(f"\n  🏷️  Key Entities:")
            if companies : print(f"     🏢 {', '.join(companies)}")
            if products  : print(f"     📦 {', '.join(products)}")
            if people    : print(f"     👤 {', '.join(people)}")

        print(f"\n  📎 References:")
        for ref in trend.get('references', []):
            print(f"     • {ref}")
        print()

print('=' * 70)


📆 WEEK-BY-WEEK TRENDS — r/TECHNOLOGY


──────────────────────────────────────────────────────────────────────
📅  Week 13 (Mar 25 – Mar 29, 2026)  |  148 posts analysed
──────────────────────────────────────────────────────────────────────

  #1 — AI's Growing Impact on Jobs and Content Creation
  The r/technology community is increasingly discussing how AI is impacting employment, with mentions of layoffs and the need for new skills. Users are also concerned about the role of AI in generating content and its implications for established platforms and industries, signaling a period of significant disruption and adaptation.

  🏷️  Key Entities:
     🏢 Meta, Epic Games, Anthropic, Microsoft, Palantir
     📦 AI, Sora
     👤 Palantir's billionaire CEO

  📎 References:
     • https://reddit.com/r/technology/comments/1s3cp2/meta_lays_off_700_employees_while_rewarding_top/
     • https://reddit.com/r/technology/comments/1s6vuih/epic_games_layoffs_included_terminally_ill_father/
     • https:/

**Cell B3 — Save Weekly Results**

In [15]:
weekly_output = {
    'generated_at'  : pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
    'model'         : GEMINI_MODEL,
    'niche'         : NICHE_WEEKLY,
    'weeks_analysed': [r['week_label'] for r in weekly_results],
    'results'       : weekly_results
}

output_file = f'{NICHE_WEEKLY}_weekly_trends.json'
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(weekly_output, f, ensure_ascii=False, indent=2)

print(f'✅ Saved to {output_file}')
print(f'   Weeks    : {len(weekly_results)}')
print(f'   Trends   : {sum(len(r["trends"]) for r in weekly_results)} total')

✅ Saved to technology_weekly_trends.json
   Weeks    : 5
   Trends   : 15 total
